# CoT Faithfulness — End-to-End Colab Run (A100)

**Before starting:** Runtime → Change runtime type → **A100 GPU**.

Store your keys in Colab secrets (the 🔑 icon in the left sidebar) with these exact names:
* `GOOGLE_API_KEY`  — Gemini, free at https://aistudio.google.com/app/apikey
* `ANTHROPIC_API_KEY` — Claude Sonnet
* `HF_TOKEN` — HuggingFace token for gated Llama 3.1 access

Total wall-clock: **~60–90 min** (mostly Gemini rate-limited + Llama attention).

## 0. Upload the repo zip and unpack
Drag `cot-faithfulness.zip` into the file pane (📁 on the left), then:

In [ ]:
!unzip -q -o cot-faithfulness.zip
%cd cot-faithfulness
!ls

## 1. Install dependencies (~3 min)

In [ ]:
!pip install -q -r requirements.txt
# Verify GPU is visible
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Load API keys from Colab secrets into environment

In [ ]:
import os
from google.colab import userdata
for k in ['GOOGLE_API_KEY', 'ANTHROPIC_API_KEY', 'HF_TOKEN']:
    try:
        os.environ[k] = userdata.get(k)
        print(f'{k} loaded ({len(os.environ[k])} chars)')
    except Exception as e:
        print(f'!! {k} missing: {e}')

## 3. Build the 75-problem subset (~1 min, downloads datasets once)

In [ ]:
!python -m scripts.make_subset

## 4. Stage 2 — Baseline CoT extraction
Runs 75 × 3 models = 225 calls.
* Llama 3.1 8B on A100: ~2 min (loads once, stays in memory)
* Gemini Flash Lite: ~5 min (free tier rate limit)
* Claude Sonnet: ~4 min

In [ ]:
# Run Llama first (uses the GPU), then the APIs.
!python -m scripts.run_baseline --models llama-3.1-8b
!python -m scripts.run_baseline --models gemini-2.5-flash-lite claude-sonnet-4-5

In [ ]:
# Quick sanity check
import pandas as pd
b = pd.read_json('results/baseline_cots.jsonl', lines=True)
print(b.groupby('model')['correct'].mean().round(3))
print(f'Well-formed CoTs: {b["well_formed"].mean():.1%}')

## 5. Stage 3 — Perturbations (~25 min)
75 × 4 perturbations × 3 models = 900 calls.

In [ ]:
!python -m scripts.run_perturbations

## 6. Stage 5 — Attention on Llama (30 problems, ~10 min on A100)
`--store-full` saves the full answer→context attention matrix so the heatmaps render in step 7.

In [ ]:
!python -m scripts.run_attention --store-full

## 7. Stage 4 — Analyze + render all figures

In [ ]:
!python -m scripts.analyze

In [ ]:
# Inline preview of headline figures
from IPython.display import Image, display
import pathlib
FIG = pathlib.Path('results/figures')
for p in sorted(FIG.glob('faithfulness_*.png')) + \
         [FIG/'f_by_difficulty.png', FIG/'convergence_scatter.png']:
    if p.exists(): display(Image(str(p)))

## 8. Download results
Saves `results/` as a zip you can pull to your laptop.

In [ ]:
!zip -qr results.zip results/
from google.colab import files
files.download('results.zip')

---
**Persistence tip:** Colab runtimes disconnect after ~90 min idle. Mount Drive and save results there:
```python
from google.colab import drive; drive.mount('/content/drive')
!cp -r results /content/drive/MyDrive/cot_faithfulness_results
```
Then if you get disconnected, you can resume — every script checkpoints to JSONL and skips what's already done.